**Step 2 of 5.** Consumes: `experimental-data/instances/qoptlib-permuted/*.{json,mps}` (step 1 output, committed). Produces: `experimental-data/results/qoptlib-mcms/{instance}_{n_subproblems}/iterations.jsonl` and `result.json` (gitignored). Runtime: worst-case ~60h (40 tasks × 1.5h each) if run sequentially; typical ~10-30h depending on convergence.

**Note:** This notebook runs the classical MCMS Benders decomposition (no quantum step) used to generate Figures 1-2 in the paper. Each task has a 1.5h time budget and will timeout gracefully without halting the kernel.

# QOptLib MCMS Benders Experiment

Runs classical MCMS (Multiple Cuts via Multiple Solutions) Benders decomposition on 10 permutations of the QOptLib XSH-n20-k4-01 instance (20 customers, 4 vehicles) with 4 different `n_subproblems` settings: {2, 4, 16, 64}.

**Total**: 40 tasks (10 instances × 4 parameter values)

**Solver configuration**:
- Master problem (MP): Cbc via PuLP
- Dual subproblems (DSP): HiGHS (returns extreme rays for feasibility cuts)
- Cut selection: Cbc via PuLP (classical minimum set cover, no quantum)

This matches the "PuLP Benders" configuration from Table 1 in the paper.

## 1. Imports and Helper Functions

In [1]:
import json
import time
import pulp
from pathlib import Path

from milp_engine.solvers.bender_milp_solver import BenderMILPSolver
from milp_engine.solvers.pulp_solver import PuLPSolver
from milp_engine.solvers.highs_solver import HiGHSSolver
from milp_engine.criteria.exclusion_criterion import ExclusionCriterion
from milp_engine.strategies.minimum_set_cover_strategy import MinimumSetCoverStrategy

from notebooks.utils import BendersSolverWithTimeout

## 2. Configuration

Define experiment parameters inline (no YAML config files).

In [2]:
# Experiment parameters (from Table 1 in paper)
INSTANCES_DIR = Path("experimental-data/instances/qoptlib-permuted")
RESULTS_DIR = Path("experimental-data/results/qoptlib-mcms")

# Benders parameters
N_SUBPROBLEMS_VALUES = [2, 4, 6, 8] # [2, 4, 16, 64]  # Parameter grid
MAX_ITERATIONS = 2500
CONVERGENCE_BOUND = 0.01
TIME_BUDGET_SECONDS = 600 # 10 minutes per task 5400  # 1.5 hours per task

# Complicating variable prefix (routing decisions)
Y_PREFIX = "x_"

print(f"Instances directory: {INSTANCES_DIR}")
print(f"Results directory: {RESULTS_DIR}")
print(f"Parameter grid: n_subproblems = {N_SUBPROBLEMS_VALUES}")
print(f"Time budget: {TIME_BUDGET_SECONDS}s ({TIME_BUDGET_SECONDS/3600:.1f}h) per task")
print(f"Total tasks: {len(list(INSTANCES_DIR.glob('*.mps'))) * len(N_SUBPROBLEMS_VALUES)}")

Instances directory: experimental-data/instances/qoptlib-permuted
Results directory: experimental-data/results/qoptlib-mcms
Parameter grid: n_subproblems = [2, 4, 6, 8]
Time budget: 600s (0.2h) per task
Total tasks: 40


## 3. Run Experiments

Loop over all (instance, n_subproblems) combinations. Each task is independent and writes to its own directory.

**Progress tracking**: Results are written to disk after each task completes, so you can interrupt and resume by skipping tasks that already have a `result.json` file.

In [3]:
# Discover instances
instance_files = sorted(INSTANCES_DIR.glob("*.mps"))
print(f"Found {len(instance_files)} instances\n")

# Build task list
tasks = []
for instance_path in instance_files[:2]: # :instance_files:
    for n_subproblems in N_SUBPROBLEMS_VALUES:
        tasks.append((instance_path, n_subproblems))

print(f"Total tasks: {len(tasks)}\n")
print("=" * 80)

Found 10 instances

Total tasks: 8



In [4]:
# Run tasks
for task_idx, (instance_path, n_subproblems) in enumerate(tasks, 1):
    instance_name = instance_path.stem
    task_name = f"{instance_name}_n{n_subproblems}"
    output_dir = RESULTS_DIR / task_name
    
    # Skip if already completed
    if (output_dir / "result.json").exists():
        print(f"[{task_idx}/{len(tasks)}] {task_name}: already completed — skipping")
        continue
    
    print(f"\n[{task_idx}/{len(tasks)}] {task_name}")
    print(f"  Instance: {instance_name}")
    print(f"  n_subproblems: {n_subproblems}")
    print(f"  Output: {output_dir}")
    
    try:
        # Load MILP from MPS file
        _, milp = pulp.LpProblem.fromMPS(instance_path)
        
        # Instantiate solver
        solver = BenderMILPSolver(
            criterion=ExclusionCriterion(),
            strategy=MinimumSetCoverStrategy(),
            mp_solver=PuLPSolver(),  # Cbc for master problem
            dsp_solver=HiGHSSolver(),  # HiGHS for dual subproblems
            cut_selection_solver=PuLPSolver(),  # Cbc for cut selection (classical)
            max_iterations=MAX_ITERATIONS,
            convergence_bound=CONVERGENCE_BOUND,
            n_subproblems=n_subproblems,
            parallel_dsp=True,
        )
        
        # Wrap with time-budget enforcement
        solver_with_timeout = BendersSolverWithTimeout(solver, TIME_BUDGET_SECONDS)
        
        # Solve
        task_start = time.time()
        solved_milp, states, exit_reason = solver_with_timeout.solve(milp, Y_PREFIX, output_dir)
        task_duration = time.time() - task_start
        
        # Print summary
        final_state = states[-1] if states else None
        print(f"  ✓ Completed: {exit_reason} after {len(states)} iterations ({task_duration:.1f}s)")
        if final_state:
            gap = BendersSolverWithTimeout._compute_gap(final_state.lower_bound, final_state.upper_bound)
            print(f"    Final bounds: LB={final_state.lower_bound:.2f}, UB={final_state.upper_bound:.2f}, gap={gap:.6f}")
    
    except Exception as e:
        print(f"  ✗ Error: {e}")
        # Write error to result file so we don't retry
        output_dir.mkdir(parents=True, exist_ok=True)
        with open(output_dir / "result.json", "w") as f:
            json.dump({"error": str(e), "exit_reason": "error"}, f)

print("\n" + "=" * 80)
print("All tasks completed!")
print(f"Results saved to {RESULTS_DIR}/")


[1/8] XSH-n20-k4-01_perm1_n2
  Instance: XSH-n20-k4-01_perm1
  n_subproblems: 2
  Output: experimental-data/results/qoptlib-mcms/XSH-n20-k4-01_perm1_n2


/zfsstore/user/valkcqde/documents/logistiqs/milp_engine/solvers/pulp_solver.py:46: UserWarning: No time limit set for the solver.
  warnings.warn("No time limit set for the solver.")
/zfsstore/user/valkcqde/documents/logistiqs/milp_engine/solvers/pulp_solver.py:51: UserWarning: No optimality gap set for the solver.
  warnings.warn("No optimality gap set for the solver.")


  Iteration 1: LB=-1.000000e+06, UB=inf, gap=inf, elapsed=0.3s
  Iteration 2: LB=-1.000000e+06, UB=inf, gap=inf, elapsed=0.5s
  Iteration 3: LB=-1.000000e+06, UB=inf, gap=inf, elapsed=0.7s
  Iteration 4: LB=-1.000000e+06, UB=inf, gap=inf, elapsed=1.0s
  Iteration 5: LB=-1.000000e+06, UB=inf, gap=inf, elapsed=1.2s
  Iteration 6: LB=-1.000000e+06, UB=inf, gap=inf, elapsed=1.5s
  Iteration 7: LB=-1.000000e+06, UB=inf, gap=inf, elapsed=1.8s
  Iteration 8: LB=-1.000000e+06, UB=inf, gap=inf, elapsed=2.0s
  Iteration 9: LB=-1.000000e+06, UB=inf, gap=inf, elapsed=2.3s
  Iteration 10: LB=-1.000000e+06, UB=inf, gap=inf, elapsed=2.7s
  Iteration 11: LB=-1.000000e+06, UB=inf, gap=inf, elapsed=3.0s
  Iteration 12: LB=-1.000000e+06, UB=inf, gap=inf, elapsed=3.2s
  Iteration 13: LB=-1.000000e+06, UB=inf, gap=inf, elapsed=3.5s
  Iteration 14: LB=-1.000000e+06, UB=inf, gap=inf, elapsed=3.8s
  Iteration 15: LB=-1.000000e+06, UB=inf, gap=inf, elapsed=4.2s
  Iteration 16: LB=-1.000000e+06, UB=inf, gap=inf

## Done

Results are in `experimental-data/results/qoptlib-mcms/` with one subdirectory per task:
- `iterations.jsonl`: Per-iteration state (bounds, gaps, runtimes, cut counts)
- `result.json`: Final summary (exit reason, total time, final bounds)

These files are consumed by notebook 05 (generate figures) to produce Figures 1-2 from the paper.